# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library from its Croissant schema.

### Dataset Source

The dataset is described and made available following the [Croissant schema standard](https://mlcommons.org/croissant/), providing a rich metadata description and efficient programmatic access.

In [ ]:
# Install mlcroissant (run this cell if mlcroissant is not installed)
!pip install mlcroissant

## 1. Data Loading

We'll load the dataset metadata and inspect its main properties using `mlcroissant`. This step will also help us identify available record sets and fields (all accessed by their `@id`).

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL to the Croissant schema JSON-LD file
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}\n")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else ''}")
print(f"Identifier: {metadata.identifier if hasattr(metadata, 'identifier') else ''}")

## 2. Data Overview

We can now discover the available record sets, their fields and columns as described in the schema, each referenced by their unique `@id` field. Listing these will help us programmatically access specific parts of the data during extraction and EDA.

In [ ]:
from pprint import pprint

print("Record Sets available in this dataset:")
record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    # Try fetching recordSet via .record_sets as in mlcroissant versions >=0.1.15
    record_sets = [rs for rs in getattr(metadata, 'record_sets', [])]

if not record_sets:
    # As a fallback, try to find them from schema
    record_sets = list(dataset.record_sets.keys()) if hasattr(dataset, 'record_sets') else []

if not record_sets:
    # Try direct access, fallback for older mlcroissant
    record_sets = list(dataset._record_sets.keys()) if hasattr(dataset, '_record_sets') else []

if not record_sets:
    print("No record sets found via metadata; querying from the Dataset object.")
    record_sets = list(dataset._record_sets.keys())

# Print the @id of each record set
for rset in record_sets:
    print(" -", rset)

# For demonstration, let's print fields for the first available record set
example_record_set_id = record_sets[0] if record_sets else None
if example_record_set_id:
    print(f"\nFields in record set {example_record_set_id}:")
    rec_set_obj = dataset._record_sets[example_record_set_id]
    field_ids = [fld['@id'] for fld in rec_set_obj['field']]
    for fld in rec_set_obj['field']:
        print(f" - {fld['@id']} (name: {fld.get('name', '')})")

## 3. Data Extraction

We will extract records from each record set as DataFrames, using proper `@id` references.

This enables flexible downstream analysis using pandas.

In [ ]:
# List of all record set @ids
record_set_ids = record_sets
dataframes = {}

# Load each record set to a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records from {record_set_id}")

# Let's examine the columns (field @ids) from the first record set
main_record_set_id = example_record_set_id

print(f"\nColumns available in main record set ({main_record_set_id}):")
if main_record_set_id and not dataframes[main_record_set_id].empty:
    pprint(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Here we'll demonstrate some typical data processing steps: filtering, normalization, and grouping, all referencing the field and record set by their `@id`.

In [ ]:
# Choose a numeric field @id for demonstration (replace with the actual @id available in your dataset)
if main_record_set_id and not dataframes[main_record_set_id].empty:
    candidate_numeric_fields = [col for col in dataframes[main_record_set_id].columns if 'age' in col.lower() or dataframes[main_record_set_id][col].dtype in ['int64', 'float64']]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
        print(f"Using numeric field (by @id): {numeric_field_id}\n")
        
        threshold = dataframes[main_record_set_id][numeric_field_id].mean() if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][numeric_field_id]) else 0
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalization
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try to group by another (categorical) field if available
        candidate_group_fields = [col for col in dataframes[main_record_set_id].columns if col != numeric_field_id and dataframes[main_record_set_id][col].nunique() < 10]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
            print(f"\nGrouping by field (by @id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
    else:
        print("No suitable numeric fields found for EDA.")
else:
    print("Main record set not found or empty.")

## 5. Visualization

Let's visualize a numeric variable's distribution and also compare grouped means, if available, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not dataframes[main_record_set_id].empty:
    # Numeric field selected in the previous section
    if 'numeric_field_id' in locals():
        plt.figure(figsize=(8, 5))
        sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()
        
        # Grouped bar plot if grouped_df exists
        if 'grouped_df' in locals() and isinstance(grouped_df, pd.Series):
            grouped_df.plot(kind='bar', figsize=(8, 4))
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()
    else:
        print("No numeric field name found for visualization.")
else:
    print("Main record set not found or empty.")

## 6. Conclusion

In this notebook, we successfully loaded, inspected, and explored the FAIR^2 dataset using the `mlcroissant` library. Each data entity was referenced by its Croissant `@id`, ensuring reproducibility and clarity.

- We explored the structure and metadata of the dataset via Croissant schema.
- Loaded the tabular data for each record set into pandas DataFrames.
- Demonstrated basic EDA tasks: filtering, normalization, and grouping, referencing all fields by their `@id`.
- Produced simple visualizations to understand the data distributions and category-wise statistics.

**You can now tailor further analysis to your specific research questions, always referencing the dataset schema for precise `@id` usage!**